# Load Dateset

In [3]:
import pandas as pd
import numpy as np

In [4]:
file_path = r"D:\Universitas Brawijaya_Roganda\BELAJAR_MANDIRI\DataScience\Project\StudentPerformanceFactors\dataset\Modeling1_StudentPerformanceFactors.csv"

df = pd.read_csv(file_path)

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Universitas Brawijaya_Roganda\\BELAJAR_MANDIRI\\DataScience\\Project\\StudentPerformanceFactors\\dataset\\Modeling1_StudentPerformanceFactors.csv'

In [ ]:
df.info()

In [ ]:
# Feature Engineering
df['Study_Efficiency'] = df['Hours_Studied'] * df['Attendance'] / 100
df['Support_Score'] = df['Parental_Involvement'] + df['Access_to_Resources'] + df['Family_Income']
df['Score_Gap'] = df['Previous_Scores'] - df['Previous_Scores'].mean()
df['Total_Support_Hours'] = df['Tutoring_Sessions'] + df['Hours_Studied']

df.info()

In [ ]:
df.head(10)

# Modeling

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Exam_Score'])
y = df['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel='rbf'),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=200, max_depth=3, random_state=42)
}

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print(results_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="R2", y="Model", palette="coolwarm")
plt.title("Model Comparison - R² Score")
plt.xlabel("R² Score")
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print(f"Best model: {best_model_name}")

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)

    plt.figure(figsize=(10,6))
    sns.barplot(x=importances.values, y=importances.index)
    plt.title(f"Feature Importance ({best_model_name})")
    plt.show()
else:
    print("Model ini gak punya feature_importances_ (biasanya linear model), cek .coef_ sebagai gantinya.")

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_ridge = {'alpha': [0.01, 0.1, 1, 10, 100]}

grid_ridge = GridSearchCV(Ridge(), param_grid_ridge, cv=5, scoring='r2')
grid_ridge.fit(X_train_scaled, y_train)

print("Best alpha (Ridge):", grid_ridge.best_params_)
print("Best CV R2:", grid_ridge.best_score_)

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0]
}

grid_xgb = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid_xgb,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_xgb.fit(X_train_scaled, y_train)

print("Best params (XGBoost):", grid_xgb.best_params_)
print("Best CV R2:", grid_xgb.best_score_)

In [ ]:
best_ridge = grid_ridge.best_estimator_
best_xgb = grid_xgb.best_estimator_

for name, model in [("Ridge (tuned)", best_ridge), ("XGBoost (tuned)", best_xgb)]:
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name} -> MAE: {mae:.3f}, RMSE: {rmse:.3f}, R2: {r2:.3f}")